# Full-range optimized potential scan with Rutherford-recoil mask

Run the revised Rutherford notebook first. It produces
`rutherford_full_range_energy_scan.npz`, containing the MB-weighted free-space
probability that the ion recoil exceeds the selected threshold.

This potential notebook then computes the trap-potential/barrier quantities and
shows its parameter-space heat maps only where

$$P_{\rm Rutherford}(E_{\rm ion} \ge E_{\rm threshold}) \ge 10^{-6}.$$

Raw potential outputs remain available. A second NPZ stores the Rutherford mask
and all masked potential metrics.

Both notebooks cover

- $10^{-27}\,\mathrm{kg} \le m_{\rm DM} \le 10^{-20}\,\mathrm{kg}$
- $10^{-3} \le \epsilon \le 10^{3}$

The speed quadrature uses $q=(0.10,0.30,0.50,0.70,0.90)$ with equal weights.


This revision also computes a speed-matched combined screening probability and plots the region recommended for full trapped-DM trajectory simulations.


In [ ]:
import numpy as np
from pathlib import Path

# ============================================================
# USER SETTINGS FOR THE FULL-RANGE POTENTIAL/ENERGY SCAN
# ============================================================

SCAN_PRESET = "balanced"  # "quick", "balanced", or "high"

M_DM_MIN_KG = 1.0e-27
M_DM_MAX_KG = 1.0e-20
EPS_MIN = 1.0e-3
EPS_MAX = 1.0e3

# (n_mass, n_epsilon, sphere directions, radial basis knots, pair chunk size)
POTENTIAL_SCAN_PRESETS = {
    "quick":    (201, 201, 64, 160, 512),
    "balanced": (401, 401, 96, 240, 512),
    "high":     (801, 801, 128, 320, 384),
}

N_MASS, N_EPS, N_DIRECTIONS, N_RADIUS_KNOTS, PAIR_CHUNK_SIZE = (
    POTENTIAL_SCAN_PRESETS[SCAN_PRESET]
)

T_DM_K = 300.0
ION_ENERGY_THRESHOLD_J = 1.0e-27
B_MAX_M = 1.0e-3

REPRESENTATIVE_SPEED_QUANTILES = np.array(
    [0.10, 0.30, 0.50, 0.70, 0.90], dtype=float
)
REPRESENTATIVE_SPEED_WEIGHTS = np.full(5, 0.2, dtype=float)

SAVE_FULL_RANGE_RESULTS = True
FULL_RANGE_RESULT_NPZ = "potential_full_range_energy_scan.npz"
FULL_RANGE_SUMMARY_CSV = "potential_full_range_energy_scan_summary.csv"


# One-way dependency: run the Rutherford notebook before this notebook.
RUTHERFORD_RESULT_NPZ = Path("rutherford_full_range_energy_scan.npz")
RUTHERFORD_RECOIL_PROBABILITY_MIN = 1.0e-6

# Combined screening estimate. A point is considered worth a full trapped
# trajectory scan when the speed-matched estimate of both (i) passing the
# potential barrier and (ii) producing threshold recoil exceeds this value.
COMBINED_SCAN_PROBABILITY_MIN = 1.0e-6
COMBINED_SCAN_HIGH_PRIORITY_MIN = 1.0e-3
COMBINED_SCAN_PROBABILITY_CONTOURS = np.array(
    [1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1], dtype=float
)
REQUIRE_RUTHERFORD_RESULTS = True
RUTHERFORD_INTERPOLATION = "log-linear"

# Save the graph-ready masked potential metrics separately from the raw outputs.
SAVE_RUTHERFORD_FILTERED_RESULTS = True
RUTHERFORD_FILTERED_RESULT_NPZ = Path(
    "potential_full_range_energy_scan_rutherford_filtered.npz"
)

# Rutherford probability contours drawn on every potential heat map.
RUTHERFORD_PROBABILITY_CONTOURS = np.array(
    [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1], dtype=float
)

# Optional straight-line Mathieu boundaries m = slope * epsilon [kg].
SHOW_MATHIEU_BOUNDARIES = True
MATHIEU_BOUNDARY_SLOPES_KG = np.array([
    7.533854e-19,
    7.561798e-24,
    7.420566e-26,
    8.939344e-27,
    8.884701e-27,
], dtype=float)

print("Potential scan preset:", SCAN_PRESET)
print(f"Grid: {N_MASS} x {N_EPS}")
print(f"Directions: {N_DIRECTIONS}; radial knots: {N_RADIUS_KNOTS}")
print("Mass range [kg]:", M_DM_MIN_KG, "to", M_DM_MAX_KG)
print("Epsilon range:", EPS_MIN, "to", EPS_MAX)

print("Rutherford mask source:", RUTHERFORD_RESULT_NPZ)
print("Rutherford probability cutoff:", RUTHERFORD_RECOIL_PROBABILITY_MIN)
print("Combined scan-probability cutoff:", COMBINED_SCAN_PROBABILITY_MIN)
print("Combined high-priority cutoff:", COMBINED_SCAN_HIGH_PRIORITY_MIN)


# Compute DC Potential

In [ ]:
## From dc_potential.ipynb
import constants as c
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
from scipy.stats import maxwell
vk = c.vk
vrf = c.vrf
z_offset = c.z0
xy1k = c.xy1k
xy2k = c.xy2k
pi = math.pi
um = c.um
cm = c.cm
mm = c.mm
K = c.K
e = c.e
z_min_valid = -c.ion_height
z_margin = 1.0e-6   # optional safety margin away from electrode plane

## Functions
# (x,y,z) = coordinate of sample
# (xik,yik,z0) = ith corner coordinates of kth electrode
# vk = voltage applied to kth electrode
def potential_term(x,y,z,xik,yik,z0):
    num = (xik-x)*(yik-y); # numerator
    den = (z-z0)*math.sqrt((z-z0)**2+(xik-x)**2+(yik-y)**2); # denominator
    return num/den

def dc_potential_single_electrode(x,y,z,x1k,y1k,x2k,y2k,z0,vk):
    term1 = math.atan(potential_term(x,y,z,x2k,y2k,z0))
    term2 = math.atan(potential_term(x,y,z,x1k,y2k,z0))
    term3 = math.atan(potential_term(x,y,z,x2k,y1k,z0))
    term4 = math.atan(potential_term(x,y,z,x1k,y1k,z0))
    return (vk/(2*math.pi))*(term1-term2-term3+term4)

def dc_potential_total(x,y,z,xy1k=c.xy1k,xy2k=c.xy2k,z0=c.z0,vk=c.vk):
    # extract individual (xik,yik),vk values from array
    # all arrays must be same size
    # sum potentials from all electrodes
    z = z+c.ion_height # account for ion height
    pot = 0
    for i in range (0,len(c.vk)):
        x1k = xy1k[i][0]
        y1k = xy1k[i][1]
        x2k = xy2k[i][0]
        y2k = xy2k[i][1]
        v = vk[i]
        if(i==19 or i==39): # make sure Q39-40 are on the M4 layer
            new_pot = dc_potential_single_electrode(x,y,z,x1k,y1k,x2k,y2k,0,v); # z0=0 for Q39-40
        else:
            new_pot = dc_potential_single_electrode(x,y,z,x1k,y1k,x2k,y2k,z0,v); # z0 = -11.8um for Q1-38
        pot+=new_pot
    return pot

# Compute RF Potential

In [ ]:
## From rf_potential.ipynb
## Functions
## Compute Pseudopotential (J)
"""
# Method 1: numerical RF gradient
Ex,Ey,Ez = np.gradient(-1*phi_rf,res,res,res); # Electric field components
E_squared = Ex**2+Ey**2+Ez**2; # Electric field strength squared
pseudo2 = (c.Z**2*c.e**2)/(4*c.m*c.omega**2)*E_squared
# pseudo_reshaped = pseudo.reshape(int(2*x_max/res+1),-1)
"""
# Method 2: analytical RF gradient
def divatan(up,down):
    #This is d(arctan2(up,down))/ddown up to a minus sign. It's useful for the pseudo-potential
    return up/(up**2+down**2)
def divatanup(up,down):
    #This is d(divatan(up,down))/dup
    return (down**2-up**2)/(up**2+down**2)**2
def divatandown(up,down):
    #This is d(divatan(up,down))/ddown
    return -2*up*down/(up**2+down**2)**2
def pseudopotential(x,y,z,m,Z,q=c.e,omrf=c.omega,VRF=c.vrf,ymin=c.y11,yedge1=c.y21,yedge2=c.y12,ymax=c.y22):
    #this is the pseudo potential, which is Z^2*(Div[PhiRF]/cos(om*t))^2/(4m*omega^2)
    z = z+c.ion_height # account for ion height
    divypart=divatan(z,yedge2-y)-divatan(z,yedge1-y)+divatan(z,ymin-y)-divatan(z,ymax-y)
    divzpart=divatan(yedge2-y,z)-divatan(yedge1-y,z)+divatan(ymin-y,z)-divatan(ymax-y,z)
    return ((Z**2*q**2)/(4*m*omrf**2)) * (VRF/math.pi)**2 * (divypart**2+divzpart**2)

In [ ]:
## Compute Full Potential Energy
def coulomb_pot(x,y,z,eps,Z=c.Z):
    return c.K*eps*Z*c.e**2/np.sqrt(x**2+y**2+z**2)
def potential_energy(xyz, m_dm, eps, component="total"):
    x, y, z = xyz[0], xyz[1], xyz[2]
    dc = eps * c.e * dc_potential_total(x, y, z)
    pseudo_rf = pseudopotential(x, y, z, m_dm, eps)
    coulomb = coulomb_pot(x, y, z, eps)
    if component == "dc":
        return dc
    elif component == "pseudo_rf":
        return pseudo_rf
    elif component == "coulomb":
        return coulomb
    elif component == "trap":
        return dc + pseudo_rf
    elif component == "total":
        return dc + pseudo_rf + coulomb
    elif component == "all":
        return dc, pseudo_rf, coulomb
    else:
        raise ValueError(f"Unknown potential component: {component}")

# Compute RF/DC Force

In [ ]:
"""
xyz : ndarray, shape (n, 3)
        Particle positions.
"""
# RF pseudo-force
def FRF(xyz,m,Z,q=c.e,omrf=c.omega,VRF=c.vrf,ymin=c.y11,yedge1=c.y21,yedge2=c.y12,ymax=c.y22):
    xyz = np.asarray(xyz, dtype=float)
    if xyz.ndim != 2 or xyz.shape[1] != 3:
        raise ValueError("xyz must have shape (n, 3)")
    x = xyz[:, 0]
    y = xyz[:, 1]
    z = xyz[:, 2] + c.ion_height # Take into account ion height
    m = float(np.asarray(m).ravel()[0])
    Z = float(np.asarray(Z).ravel()[0])
    q = float(np.asarray(q).ravel()[0])
    omrf = float(np.asarray(omrf).ravel()[0])
    VRF = float(np.asarray(VRF).ravel()[0])
    #this is the pseudo potential, which is Z^2*(Div[PhiRF]/cos(om*t))^2/(4m*omega^2)
    divypart=divatan(z,yedge2-y)-divatan(z,yedge1-y)+divatan(z,ymin-y)-divatan(z,ymax-y)
    divzpart=divatan(yedge2-y,z)-divatan(yedge1-y,z)+divatan(ymin-y,z)-divatan(ymax-y,z) 
    divyparty=-2*divypart*(divatandown(z,yedge2-y)-divatandown(z,yedge1-y)+divatandown(z,ymin-y)-divatandown(z,ymax-y)) 
    divypartz=2*divypart*(divatanup(z,yedge2-y)-divatanup(z,yedge1-y)+divatanup(z,ymin-y)-divatanup(z,ymax-y)) 
    divzparty=-2*divzpart*(divatanup(yedge2-y,z)-divatanup(yedge1-y,z)+divatanup(ymin-y,z)-divatanup(ymax-y,z)) 
    divzpartz=2*divzpart*(divatandown(yedge2-y,z)-divatandown(yedge1-y,z)+divatandown(ymin-y,z)-divatandown(ymax-y,z)) 
    return -1*((VRF*Z*q)/(2*pi*np.sqrt(m)*omrf))**2*np.column_stack([divypartz*0, divzparty+divyparty, divzpartz+divypartz])

# DC force
def divatan(up,down):
    #This is d(arctan2(up,down))/ddown up to a minus sign. It's useful for the pseudo-potential
    return up/(up**2+down**2)
def divatanup(up,down):
    #This is d(divatan(up,down))/dup
    return (down**2-up**2)/(up**2+down**2)**2
def divatandown(up,down):
    #This is d(divatan(up,down))/ddown
    return -2*up*down/(up**2+down**2)**2

def anatangrad(xi,yi,xyz,v): # gradient term of DC potential
    xyz = np.asarray(xyz, dtype=float)
    x = xyz[:, 0]
    y = xyz[:, 1]
    z = xyz[:, 2]
    dy=y-yi
    dx=x-xi
    r = np.sqrt(dx**2+dy**2+z**2); # added distance
    dry2=z**2+dy**2
    drx2=z**2+dx**2
    divy=z*dx/(r*dry2); # divide by factor r
    divz=-dy*dx*(1/dry2+1/drx2)/r; # divide by factor r
    divx=z*dy/(r*drx2); # divide by factor r
    return (v/(2*np.pi))*np.column_stack([divx, divy, divz])

def FDC_single(x1,y1,x2,y2,xyz,v,Z,q=c.e):
    return -Z*q * (anatangrad(x2,y2,xyz,v)-anatangrad(x2,y1,xyz,v)-anatangrad(x1,y2,xyz,v)+anatangrad(x1,y1,xyz,v))

def FDC(xyz, m, Z, q=c.e, omrf=c.omega,
        ymin=c.y11, yedge1=c.y21, yedge2=c.y12, ymax=c.y22):
    xyz = np.asarray(xyz, dtype=float)
    if xyz.ndim != 2 or xyz.shape[1] != 3:
        raise ValueError("xyz must have shape (n, 3)")
    z_original = xyz[:, 2].copy() + c.ion_height # Take into account ion height
    force = np.zeros_like(xyz)
    for k in range(0, 40):
        xyz_k = xyz.copy()
        if (k != 19 and k != 39):
            xyz_k[:, 2] = z_original - z_offset  # keep scalar math
        else:
            xyz_k[:, 2] = z_original
        (x1k, y1k) = xy1k[k]
        (x2k, y2k) = xy2k[k]
        # enforce scalar on vk[k]
        v_k = float(np.asarray(vk[k]).ravel()[0])
        force_single = FDC_single(x1k, y1k, x2k, y2k, xyz_k, v_k, Z, q=q)
        # check shape of force_single 
        force_single = np.asarray(force_single, dtype=float)
        if force_single.shape != xyz.shape:
            raise ValueError(
                f"FDC_single returned unexpected shape {force_single.shape}; "
                f"expected {xyz.shape} at k={k}"
            )
        force += force_single
    return force

def force(xyz, m_dm, eps, component="total"):
    xyz = np.asarray(xyz, dtype=float)

    single_input = False
    if xyz.ndim == 1:
        xyz = xyz.reshape(1, 3)
        single_input = True

    if xyz.ndim != 2 or xyz.shape[1] != 3:
        raise ValueError("xyz must have shape (n, 3) or (3,)")

    dc_force = FDC(xyz, m_dm, eps)
    pseudo_force = FRF(xyz, m_dm, eps)

    r = np.linalg.norm(xyz, axis=1)

    coulomb_force = np.full_like(xyz, np.nan, dtype=float)
    good = r > 0

    coulomb_force[good] = (
        c.K * c.Z * eps * c.e**2
        * xyz[good]
        / r[good, None]**3
    )

    if component == "dc":
        out = dc_force
    elif component == "pseudo_rf":
        out = pseudo_force
    elif component == "coulomb":
        out = coulomb_force
    elif component == "trap":
        out = dc_force + pseudo_force
    elif component == "total":
        out = dc_force + pseudo_force + coulomb_force
    elif component == "all":
        return dc_force, pseudo_force, coulomb_force
    else:
        raise ValueError(f"Unknown force component: {component}")

    if single_input:
        return out[0]

    return out

# Test Energy at r_min_threshold for (m_dm,eps)

In [ ]:
def xyz_on_sphere(n, radius, center=(0.0, 0.0, 0.0)):
    """
    Generate positions evenly distributed on the surface of a sphere.

    Parameters
    ----------
    n_dm : int
        Number of coordinate points.
    radius : float
        Sphere radius in meters.
    center : array-like, shape (3,)
        Sphere center in meters.
    
    Returns
    -------
    x0_dms : ndarray, shape (n_dm, 3)
        Positions on the sphere surface.
    """

    center = np.asarray(center, dtype=float)
    if n <= 0:
        raise ValueError("n_dm must be positive")
    indices = np.arange(n)
    golden_angle = np.pi * (3.0 - np.sqrt(5.0))
    z = 1.0 - 2.0 * (indices + 0.5) / n
    r_xy = np.sqrt(1.0 - z**2)
    theta = golden_angle * indices
    x = r_xy * np.cos(theta)
    y = r_xy * np.sin(theta)
    points = np.column_stack([x, y, z])
    x0_dms = center + radius * points
    return x0_dms

def valid_xyz_mask(xyz):
    xyz = np.asarray(xyz, dtype=float)
    return xyz[..., 2] > (z_min_valid + z_margin)

def xyz_on_valid_sphere(n, radius, center=(0.0, 0.0, 0.0)):
    xyz = xyz_on_sphere(n, radius, center=center)
    mask = valid_xyz_mask(xyz)
    return xyz[mask]

def valid_fraction_on_sphere(n, radius):
    xyz = xyz_on_sphere(n, radius)
    return np.mean(valid_xyz_mask(xyz))

def PE_on_sphere(n, radius, m_dm, eps, component="total"):
    xyz = xyz_on_valid_sphere(n, radius)
    PE_list = []
    for row in xyz:
        PE_list.append(potential_energy(row, m_dm, eps, component))
    return PE_list

def force_on_sphere(n, radius, m_dm, eps, component="total"):
    xyz = xyz_on_valid_sphere(n, radius)
    F_list = []
    for row in xyz:
        F_list.append(force([row], m_dm, eps, component))
    return F_list

def deterministic_MB_speeds(T, m_dm, quantile):
    k_B = 1.380649e-23
    scale = np.sqrt(k_B * T / m_dm)
    speeds = maxwell.ppf(quantile, scale=scale)
    return speeds

# Test Consistency of Potential vs Force

In [ ]:
m_dm = 2.636651e-25
eps = 2.335721

def numerical_force_from_potential(xyz, m_dm, eps, component, h=0.05e-6):
    xyz = np.asarray(xyz, dtype=float)

    if xyz.shape != (3,):
        raise ValueError("xyz must have shape (3,)")

    F = np.zeros(3)

    for k in range(3):
        step = np.zeros(3)
        step[k] = h

        xyz_plus = xyz + step
        xyz_minus = xyz - step

        # Avoid finite-difference points outside valid potential domain.
        if xyz_plus[2] <= z_min_valid + z_margin:
            F[k] = np.nan
            continue

        if xyz_minus[2] <= z_min_valid + z_margin:
            F[k] = np.nan
            continue

        U_plus = potential_energy(xyz_plus, m_dm, eps, component)
        U_minus = potential_energy(xyz_minus, m_dm, eps, component)

        F[k] = -(U_plus - U_minus) / (2.0 * h)

    return F


test_points = [
    np.array([0.0, 0.0, 1e-6]),
    np.array([0.0, 0.0, 10e-6]),
    np.array([10e-6, 0.0, 10e-6]),
    np.array([0.0, 10e-6, 10e-6]),
    np.array([0.0, 0.0, 100e-6]),
    np.array([100e-6, 0.0, 300e-6]),
]

for xyz in test_points:
    if xyz[2] <= z_min_valid + z_margin:
        continue

    print()
    print("xyz =", xyz)

    for component in ["dc", "pseudo_rf", "coulomb", "trap", "total"]:
        F_analytic = np.asarray(force(xyz, m_dm, eps, component), dtype=float).reshape(3)
        F_numeric = numerical_force_from_potential(xyz, m_dm, eps, component)

        diff = F_analytic - F_numeric

        denom = max(
            np.linalg.norm(F_analytic),
            np.linalg.norm(F_numeric),
            1e-300,
        )

        rel_err = np.linalg.norm(diff) / denom

        print(component)
        print("  analytic:", F_analytic)
        print("  numeric :", F_numeric)
        print("  diff    :", diff)
        print("  rel_err :", rel_err)

# Optimized full-range potential-barrier and ion-energy scan

The electrode geometry is precomputed only on a compact radial/directional basis.
The full $(m_{m DM},\epsilon)$ grid then uses logarithmic radial interpolation,
chunked vectorized algebra, and analytic free-space Rutherford threshold radii.
No hard-coded 20x20 threshold table is used.


In [ ]:
import time
from scipy.stats import maxwell

K_B = 1.380649e-23
E_threshold = ION_ENERGY_THRESHOLD_J
T_dm = T_DM_K


def _fibonacci_directions(n):
    i = np.arange(int(n), dtype=float)
    z = 1.0 - 2.0 * (i + 0.5) / n
    rho = np.sqrt(np.maximum(0.0, 1.0 - z*z))
    phi = np.pi * (3.0 - np.sqrt(5.0)) * i
    return np.column_stack((rho*np.cos(phi), rho*np.sin(phi), z))


def _dc_potential_vectorized(xyz):
    """Vectorized equivalent of dc_potential_total for many positions."""
    xyz = np.asarray(xyz, dtype=float).reshape(-1, 3)
    x = xyz[:, 0]
    y = xyz[:, 1]
    z_shift = xyz[:, 2] + c.ion_height
    total = np.zeros(x.size, dtype=float)

    def term(xi, yi, dz):
        dx = xi - x
        dy = yi - y
        numerator = dx * dy
        denominator = dz * np.sqrt(dz*dz + dx*dx + dy*dy)
        ratio = np.divide(
            numerator, denominator,
            out=np.zeros_like(numerator), where=denominator != 0.0,
        )
        return np.arctan(ratio)

    for index in range(len(c.vk)):
        x1, y1 = c.xy1k[index]
        x2, y2 = c.xy2k[index]
        layer_z = 0.0 if index in (19, 39) else c.z0
        dz = z_shift - layer_z
        total += float(np.asarray(c.vk[index]).ravel()[0]) / (2.0*np.pi) * (
            term(x2, y2, dz) - term(x1, y2, dz)
            - term(x2, y1, dz) + term(x1, y1, dz)
        )
    return total


def _pseudo_basis_vectorized(xyz):
    """Return U_pseudo for m=1 kg and epsilon=1; scale by eps^2/m later."""
    xyz = np.asarray(xyz, dtype=float).reshape(-1, 3)
    y = xyz[:, 1]
    z = xyz[:, 2] + c.ion_height
    div_y = (
        divatan(z, c.y12-y) - divatan(z, c.y21-y)
        + divatan(z, c.y11-y) - divatan(z, c.y22-y)
    )
    div_z = (
        divatan(c.y12-y, z) - divatan(c.y21-y, z)
        + divatan(c.y11-y, z) - divatan(c.y22-y, z)
    )
    prefactor = c.e**2 / (4.0 * c.omega**2) * (c.vrf/np.pi)**2
    return prefactor * (div_y*div_y + div_z*div_z)


def _analytic_rutherford_threshold_radius(mass, epsilon, speed_unit):
    """Largest closest approach whose free-space recoil reaches threshold."""
    mass = np.asarray(mass, dtype=float)
    epsilon = np.asarray(epsilon, dtype=float)
    v2 = speed_unit**2 * K_B * T_DM_K / mass
    mu = mass * c.m / (mass + c.m)
    E_rel = 0.5 * mu * v2
    C = c.K * abs(c.Z) * c.e**2 * epsilon
    a = C / (2.0 * E_rel)
    transfer = 4.0 * mass * c.m / (mass + c.m)**2
    K_incident = 0.5 * speed_unit**2 * K_B * T_DM_K
    E_max = transfer * K_incident
    y_total = B_MAX_M**2 / np.maximum(a*a, np.finfo(float).tiny)
    y_detect = np.clip(E_max / E_threshold - 1.0, 0.0, y_total)
    radius = a * (1.0 + np.sqrt(1.0 + y_detect))
    return np.where(y_detect > 0.0, radius, np.nan)


def _analytic_rutherford_detection_probability(mass, epsilon, speed_unit):
    """Free-space impact-parameter probability of threshold ion recoil.

    The impact parameter is uniform in area over 0 <= b <= B_MAX_M.  This is
    the same analytic Rutherford model and speed discretization used by the
    coupled Rutherford notebook.
    """
    mass = np.asarray(mass, dtype=float)
    epsilon = np.asarray(epsilon, dtype=float)
    v2 = speed_unit**2 * K_B * T_DM_K / mass
    mu = mass * c.m / (mass + c.m)
    E_rel = 0.5 * mu * v2
    C = c.K * abs(c.Z) * c.e**2 * epsilon
    a = C / (2.0 * E_rel)

    transfer = 4.0 * mass * c.m / (mass + c.m)**2
    K_incident = 0.5 * speed_unit**2 * K_B * T_DM_K
    E_max = transfer * K_incident

    y_total = B_MAX_M**2 / np.maximum(a*a, np.finfo(float).tiny)
    y_detect = np.clip(E_max / E_threshold - 1.0, 0.0, y_total)
    return np.divide(
        y_detect, y_total,
        out=np.zeros(np.broadcast_shapes(mass.shape, epsilon.shape), dtype=float),
        where=y_total > 0.0,
    )


def _precompute_radial_potential_basis(radius_knots, directions):
    """Expensive electrode geometry is evaluated only on this small basis."""
    n_r = radius_knots.size
    n_d = directions.shape[0]
    dc_basis = np.full((n_r, n_d), np.nan, dtype=np.float64)
    pseudo_basis = np.full((n_r, n_d), np.nan, dtype=np.float64)
    valid_basis = np.zeros((n_r, n_d), dtype=bool)

    for ir, radius in enumerate(radius_knots):
        xyz = radius * directions
        valid = xyz[:, 2] > (z_min_valid + z_margin)
        valid_basis[ir] = valid
        if np.any(valid):
            dc_basis[ir, valid] = _dc_potential_vectorized(xyz[valid])
            pseudo_basis[ir, valid] = _pseudo_basis_vectorized(xyz[valid])
        if ir == 0 or (ir + 1) % max(1, n_r//10) == 0 or ir + 1 == n_r:
            print(f"radial potential basis {ir+1:4d}/{n_r}")
    return dc_basis, pseudo_basis, valid_basis


def _interpolate_radial_basis(radii, radius_knots, dc_basis, pseudo_basis, directions):
    radii = np.asarray(radii, dtype=float).reshape(-1)
    n = radii.size
    n_d = directions.shape[0]
    dc = np.full((n, n_d), np.nan, dtype=float)
    pseudo = np.full((n, n_d), np.nan, dtype=float)
    direct_valid = np.zeros((n, n_d), dtype=bool)

    good = np.isfinite(radii) & (radii > 0.0)
    if not np.any(good):
        return dc, pseudo, direct_valid

    log_knots = np.log(radius_knots)
    log_r = np.log(np.clip(radii[good], radius_knots[0], radius_knots[-1]))
    right = np.searchsorted(log_knots, log_r, side="right")
    right = np.clip(right, 1, len(radius_knots)-1)
    left = right - 1
    w = (log_r - log_knots[left]) / (log_knots[right] - log_knots[left])

    def blend(table):
        a = table[left]
        b = table[right]
        both = np.isfinite(a) & np.isfinite(b)
        only_a = np.isfinite(a) & ~np.isfinite(b)
        only_b = ~np.isfinite(a) & np.isfinite(b)
        out = np.full_like(a, np.nan)
        out[both] = ((1.0-w)[:, None]*a + w[:, None]*b)[both]
        out[only_a] = a[only_a]
        out[only_b] = b[only_b]
        return out

    dc_good = blend(dc_basis)
    pseudo_good = blend(pseudo_basis)
    valid_good = radii[good, None] * directions[None, :, 2] > (z_min_valid + z_margin)
    dc_good[~valid_good] = np.nan
    pseudo_good[~valid_good] = np.nan

    dc[good] = dc_good
    pseudo[good] = pseudo_good
    direct_valid[good] = valid_good & np.isfinite(dc_good) & np.isfinite(pseudo_good)
    return dc, pseudo, direct_valid


def _safe_nanquantile(values, q):
    values = np.asarray(values, dtype=float)
    out = np.full(values.shape[0], np.nan, dtype=float)
    good_rows = np.any(np.isfinite(values), axis=1)
    if np.any(good_rows):
        with np.errstate(all="ignore"):
            out[good_rows] = np.nanquantile(values[good_rows], q, axis=1)
    return out


start = time.perf_counter()
m_dm_values = np.geomspace(M_DM_MIN_KG, M_DM_MAX_KG, N_MASS)
eps_values = np.geomspace(EPS_MIN, EPS_MAX, N_EPS)
EPS_grid, M_grid = np.meshgrid(eps_values, m_dm_values)
mass_flat = M_grid.ravel()
eps_flat = EPS_grid.ravel()
n_pair = mass_flat.size

speed_units = maxwell.ppf(REPRESENTATIVE_SPEED_QUANTILES)
directions = _fibonacci_directions(N_DIRECTIONS)

# Determine the radial basis range from all representative threshold radii.
radius_min = np.inf
radius_max = 0.0
for speed_unit in speed_units:
    r = _analytic_rutherford_threshold_radius(M_grid, EPS_grid, speed_unit)
    finite = r[np.isfinite(r) & (r > 0.0)]
    if finite.size:
        radius_min = min(radius_min, float(np.min(finite)))
        radius_max = max(radius_max, float(np.max(finite)))
if not np.isfinite(radius_min) or radius_max <= 0.0:
    raise RuntimeError("No free-space Rutherford threshold radii exist on this grid")
radius_min = max(radius_min * 0.8, 1.0e-9)
radius_max = radius_max * 1.2
radius_knots = np.geomspace(radius_min, radius_max, N_RADIUS_KNOTS)
print(f"Radial basis: {radius_min*1e6:.6g} to {radius_max*1e6:.6g} um")

dc_basis, pseudo_basis, valid_basis = _precompute_radial_potential_basis(
    radius_knots, directions
)

shape = (N_MASS, N_EPS)
output_names = [
    "barrier_access_probability",
    "ion_upper_probability_ge_threshold",
    "ion_upper_unconditional_mean_J",
    "ion_upper_conditional_mean_J",
    "ion_upper_conditional_median_J",
    "ion_upper_conditional_p90_J",
    "ion_upper_q10_fullsphere_mean_J",
    "ion_upper_q10_probability_ge_threshold",
    "q10_r_min_threshold_m",
    "valid_direction_fraction_q10",
    "rutherford_probability_recomputed",
    "combined_scan_probability_estimate",
]
outputs = {name: np.full(n_pair, np.nan, dtype=np.float32) for name in output_names}

for i0 in range(0, n_pair, PAIR_CHUNK_SIZE):
    i1 = min(i0 + PAIR_CHUNK_SIZE, n_pair)
    mass = mass_flat[i0:i1]
    epsilon = eps_flat[i0:i1]
    n_chunk = i1 - i0
    transfer = 4.0 * mass * c.m / (mass + c.m)**2

    all_energy_events = []
    all_detected_events = []
    access_probability_q = []
    threshold_probability_q = []
    mean_q = []
    rutherford_probability_q = []
    q10_radius = None
    q10_valid_fraction = None

    for iq, speed_unit in enumerate(speed_units):
        radius = _analytic_rutherford_threshold_radius(mass, epsilon, speed_unit)
        dc, pseudo, valid = _interpolate_radial_basis(
            radius, radius_knots, dc_basis, pseudo_basis, directions
        )

        coulomb_per_eps = c.K * abs(c.Z) * c.e**2 / radius[:, None]
        linear_per_eps = c.e * dc + coulomb_per_eps
        U = epsilon[:, None] * linear_per_eps + (
            epsilon[:, None]**2 / mass[:, None]
        ) * pseudo

        K_inf = 0.5 * speed_unit**2 * K_B * T_DM_K
        K_local = np.maximum(K_inf - U, 0.0)
        E_upper = transfer[:, None] * K_local
        E_upper[~valid] = 0.0

        access = valid & (K_local > 0.0)
        detected = valid & (E_upper >= E_threshold)
        access_probability_q.append(np.mean(access, axis=1))
        threshold_probability_q.append(np.mean(detected, axis=1))
        mean_q.append(np.mean(E_upper, axis=1))
        rutherford_probability_q.append(
            _analytic_rutherford_detection_probability(
                mass, epsilon, speed_unit
            )
        )

        all_energy_events.append(E_upper)
        all_detected_events.append(np.where(detected, E_upper, np.nan))

        if iq == 0:
            q10_radius = radius
            q10_valid_fraction = np.mean(valid, axis=1)

    energy_events = np.concatenate(all_energy_events, axis=1)
    detected_events = np.concatenate(all_detected_events, axis=1)
    access_stack = np.stack(access_probability_q)
    threshold_stack = np.stack(threshold_probability_q)
    mean_stack = np.stack(mean_q)
    rutherford_probability_stack = np.stack(rutherford_probability_q)

    barrier_access_probability = np.sum(
        REPRESENTATIVE_SPEED_WEIGHTS[:, None] * access_stack, axis=0
    )
    threshold_probability = np.sum(
        REPRESENTATIVE_SPEED_WEIGHTS[:, None] * threshold_stack, axis=0
    )
    unconditional_mean = np.sum(
        REPRESENTATIVE_SPEED_WEIGHTS[:, None] * mean_stack, axis=0
    )

    # Speed-matched combined screening estimate. At each representative MB
    # speed, impact parameter and incoming direction are treated as independent.
    # This avoids the less meaningful product of two separately speed-averaged
    # probabilities.
    rutherford_probability_recomputed = np.sum(
        REPRESENTATIVE_SPEED_WEIGHTS[:, None]
        * rutherford_probability_stack,
        axis=0,
    )
    combined_scan_probability_estimate = np.sum(
        REPRESENTATIVE_SPEED_WEIGHTS[:, None]
        * rutherford_probability_stack
        * access_stack,
        axis=0,
    )

    detected_count = np.sum(np.isfinite(detected_events), axis=1)
    detected_sum = np.nansum(detected_events, axis=1)
    conditional_mean = np.divide(
        detected_sum, detected_count,
        out=np.full(n_chunk, np.nan), where=detected_count > 0,
    )
    conditional_median = _safe_nanquantile(detected_events, 0.50)
    conditional_p90 = _safe_nanquantile(detected_events, 0.90)

    chunk_values = {
        "barrier_access_probability": barrier_access_probability,
        "ion_upper_probability_ge_threshold": threshold_probability,
        "ion_upper_unconditional_mean_J": unconditional_mean,
        "ion_upper_conditional_mean_J": conditional_mean,
        "ion_upper_conditional_median_J": conditional_median,
        "ion_upper_conditional_p90_J": conditional_p90,
        "ion_upper_q10_fullsphere_mean_J": mean_stack[0],
        "ion_upper_q10_probability_ge_threshold": threshold_stack[0],
        "q10_r_min_threshold_m": q10_radius,
        "valid_direction_fraction_q10": q10_valid_fraction,
        "rutherford_probability_recomputed": (
            rutherford_probability_recomputed
        ),
        "combined_scan_probability_estimate": (
            combined_scan_probability_estimate
        ),
    }
    for name, value in chunk_values.items():
        outputs[name][i0:i1] = np.asarray(value, dtype=np.float32)

    if i0 == 0 or i1 == n_pair or (i0 // PAIR_CHUNK_SIZE) % 20 == 0:
        print(f"parameter pairs {i0:7d}:{i1:7d} / {n_pair}")

for name in outputs:
    outputs[name] = outputs[name].reshape(shape)

# Compatibility aliases for the earlier notebook's plotting conventions.
r_min_arr = outputs["q10_r_min_threshold_m"].astype(float)
P_area_full_grid = outputs["barrier_access_probability"].astype(float)
P_easiest_grid = np.nanmax(
    np.stack([
        outputs["barrier_access_probability"],
        outputs["ion_upper_probability_ge_threshold"],
    ]), axis=0,
)

runtime_s = time.perf_counter() - start
print(f"Optimized full-range potential scan finished in {runtime_s:.2f} s")
print("Lowest representative speed quantile: q=0.10")
finite_combined = outputs["combined_scan_probability_estimate"]
worth_count = int(np.count_nonzero(
    np.isfinite(finite_combined)
    & (finite_combined >= COMBINED_SCAN_PROBABILITY_MIN)
))
high_count = int(np.count_nonzero(
    np.isfinite(finite_combined)
    & (finite_combined >= COMBINED_SCAN_HIGH_PRIORITY_MIN)
))
print(
    "Combined screening points worth scanning: "
    f"{worth_count}/{finite_combined.size} "
    f"({100.0*worth_count/finite_combined.size:.3f}%)"
)
print(
    "Combined high-priority points: "
    f"{high_count}/{finite_combined.size} "
    f"({100.0*high_count/finite_combined.size:.3f}%)"
)

if SAVE_FULL_RANGE_RESULTS:
    np.savez_compressed(
        FULL_RANGE_RESULT_NPZ,
        m_dm_values=m_dm_values,
        eps_values=eps_values,
        radius_knots=radius_knots,
        speed_quantiles=REPRESENTATIVE_SPEED_QUANTILES,
        speed_weights=REPRESENTATIVE_SPEED_WEIGHTS,
        runtime_s=runtime_s,
        **outputs,
    )
    summary = pd.DataFrame({
        "metric": list(outputs),
        "finite_count": [int(np.count_nonzero(np.isfinite(outputs[k]))) for k in outputs],
        "minimum": [float(np.nanmin(outputs[k])) if np.any(np.isfinite(outputs[k])) else np.nan for k in outputs],
        "maximum": [float(np.nanmax(outputs[k])) if np.any(np.isfinite(outputs[k])) else np.nan for k in outputs],
    })
    summary.to_csv(FULL_RANGE_SUMMARY_CSV, index=False)
    print("Saved", FULL_RANGE_RESULT_NPZ)
    print("Saved", FULL_RANGE_SUMMARY_CSV)


# Load Rutherford recoil probability and build the plotting mask

The Rutherford and potential scans may use different grid resolutions. The code
therefore validates the physical settings, then log-linearly interpolates the
Rutherford probability onto the potential grid. No potential metric is used to
construct this mask.


In [ ]:
from scipy.interpolate import RegularGridInterpolator


def _npz_scalar(dataset, key):
    if key not in dataset.files:
        return None
    value = np.asarray(dataset[key])
    if value.size != 1:
        raise ValueError(f"Expected scalar metadata {key!r}, got shape {value.shape}")
    return value.reshape(()).item()


def _check_matching_scalar(dataset, key, expected, *, rtol=1e-10, atol=0.0):
    actual = _npz_scalar(dataset, key)
    if actual is None:
        print(f"Warning: legacy Rutherford file has no {key!r} metadata.")
        return
    if not np.isclose(float(actual), float(expected), rtol=rtol, atol=atol):
        raise ValueError(
            f"Rutherford result mismatch for {key}: file has {actual!r}, "
            f"but potential notebook uses {expected!r}. Rerun Rutherford first."
        )


def _load_rutherford_probability_on_potential_grid(path):
    path = Path(path)
    if not path.exists():
        message = (
            f"Missing Rutherford result file: {path.resolve()}\n"
            "Run truncate_dm_parameters_rutherford_full_range_coupled.ipynb "
            "first, then rerun this notebook."
        )
        if REQUIRE_RUTHERFORD_RESULTS:
            raise FileNotFoundError(message)
        print("Warning:", message)
        return np.full(M_grid.shape, np.nan), np.zeros(M_grid.shape, dtype=bool)

    with np.load(path, allow_pickle=False) as dataset:
        required = {"m_dm_values", "eps_values", "prob_E_above_threshold"}
        missing = sorted(required.difference(dataset.files))
        if missing:
            raise KeyError(f"Rutherford NPZ is missing required arrays: {missing}")

        ruth_m = np.asarray(dataset["m_dm_values"], dtype=float).reshape(-1)
        ruth_eps = np.asarray(dataset["eps_values"], dtype=float).reshape(-1)
        ruth_prob = np.asarray(dataset["prob_E_above_threshold"], dtype=float)

        _check_matching_scalar(dataset, "temperature_K", T_DM_K)
        _check_matching_scalar(
            dataset, "ion_energy_threshold_J", ION_ENERGY_THRESHOLD_J,
            rtol=1e-10, atol=max(1e-40, abs(ION_ENERGY_THRESHOLD_J)*1e-12),
        )
        _check_matching_scalar(dataset, "b_max_m", B_MAX_M)

        if "speed_quantiles" in dataset.files and not np.allclose(
            np.asarray(dataset["speed_quantiles"], dtype=float),
            REPRESENTATIVE_SPEED_QUANTILES,
            rtol=0.0, atol=1e-14,
        ):
            raise ValueError("Rutherford and potential speed quantiles do not match.")
        if "speed_weights" in dataset.files and not np.allclose(
            np.asarray(dataset["speed_weights"], dtype=float),
            REPRESENTATIVE_SPEED_WEIGHTS,
            rtol=0.0, atol=1e-14,
        ):
            raise ValueError("Rutherford and potential speed weights do not match.")

    if ruth_prob.shape != (ruth_m.size, ruth_eps.size):
        raise ValueError(
            "Rutherford probability shape mismatch: "
            f"{ruth_prob.shape} versus {(ruth_m.size, ruth_eps.size)}"
        )
    if np.any(ruth_m <= 0.0) or np.any(ruth_eps <= 0.0):
        raise ValueError("Rutherford mass and epsilon grids must be positive.")

    m_order = np.argsort(ruth_m)
    e_order = np.argsort(ruth_eps)
    ruth_m = ruth_m[m_order]
    ruth_eps = ruth_eps[e_order]
    ruth_prob = ruth_prob[np.ix_(m_order, e_order)]

    if (
        ruth_prob.shape == M_grid.shape
        and np.allclose(ruth_m, m_dm_values, rtol=1e-12, atol=0.0)
        and np.allclose(ruth_eps, eps_values, rtol=1e-12, atol=0.0)
    ):
        probability = ruth_prob.copy()
        interpolation_used = "none; grids matched exactly"
    else:
        if RUTHERFORD_INTERPOLATION != "log-linear":
            raise ValueError("RUTHERFORD_INTERPOLATION must be 'log-linear'.")
        # Probability changes by many orders of magnitude, so interpolate log10(P).
        log_probability = np.log10(np.clip(ruth_prob, 1e-300, 1.0))
        interpolator = RegularGridInterpolator(
            (np.log10(ruth_m), np.log10(ruth_eps)),
            log_probability,
            method="linear",
            bounds_error=False,
            fill_value=np.nan,
        )
        query = np.column_stack((
            np.log10(M_grid.ravel()),
            np.log10(EPS_grid.ravel()),
        ))
        probability = 10.0 ** interpolator(query).reshape(M_grid.shape)
        interpolation_used = "log-linear in (log10 m, log10 epsilon, log10 P)"

    probability = np.clip(probability, 0.0, 1.0)
    keep = (
        np.isfinite(probability)
        & (probability >= RUTHERFORD_RECOIL_PROBABILITY_MIN)
    )
    print("Loaded Rutherford mask from:", path.resolve())
    print("Rutherford interpolation:", interpolation_used)
    print(
        "Potential-grid points retained by Rutherford cutoff: "
        f"{np.count_nonzero(keep)}/{keep.size} "
        f"({100.0*np.mean(keep):.3f}%)"
    )
    return probability.astype(np.float32), keep


(
    rutherford_probability_on_potential_grid,
    rutherford_keep_mask,
) = _load_rutherford_probability_on_potential_grid(RUTHERFORD_RESULT_NPZ)

# Validate that the locally recomputed speed-resolved Rutherford model agrees
# with the Rutherford notebook after interpolation onto this grid.
local_rutherford_probability = np.asarray(
    outputs["rutherford_probability_recomputed"], dtype=float
)
comparison = (
    np.isfinite(local_rutherford_probability)
    & np.isfinite(rutherford_probability_on_potential_grid)
)
if np.any(comparison):
    absolute_difference = np.abs(
        local_rutherford_probability[comparison]
        - rutherford_probability_on_potential_grid[comparison]
    )
    print(
        "Max local-vs-loaded Rutherford probability difference:",
        f"{np.max(absolute_difference):.3e}",
    )

combined_scan_probability = np.asarray(
    outputs["combined_scan_probability_estimate"], dtype=float
)
combined_scan_keep_mask = (
    rutherford_keep_mask
    & np.isfinite(combined_scan_probability)
    & (combined_scan_probability >= COMBINED_SCAN_PROBABILITY_MIN)
)
combined_scan_high_priority_mask = (
    combined_scan_keep_mask
    & (combined_scan_probability >= COMBINED_SCAN_HIGH_PRIORITY_MIN)
)
print(
    "Combined estimate points worth a full scan: "
    f"{np.count_nonzero(combined_scan_keep_mask)}/{combined_scan_keep_mask.size} "
    f"({100.0*np.mean(combined_scan_keep_mask):.3f}%)"
)
print(
    "Combined high-priority points: "
    f"{np.count_nonzero(combined_scan_high_priority_mask)}/"
    f"{combined_scan_high_priority_mask.size} "
    f"({100.0*np.mean(combined_scan_high_priority_mask):.3f}%)"
)

filtered_outputs = {
    name: np.where(rutherford_keep_mask, np.asarray(value, dtype=float), np.nan).astype(np.float32)
    for name, value in outputs.items()
}

if SAVE_RUTHERFORD_FILTERED_RESULTS:
    filtered_payload = {
        "m_dm_values": m_dm_values,
        "eps_values": eps_values,
        "temperature_K": np.asarray(T_DM_K),
        "ion_energy_threshold_J": np.asarray(ION_ENERGY_THRESHOLD_J),
        "b_max_m": np.asarray(B_MAX_M),
        "rutherford_probability_cutoff": np.asarray(
            RUTHERFORD_RECOIL_PROBABILITY_MIN
        ),
        "rutherford_prob_E_above_threshold": (
            rutherford_probability_on_potential_grid
        ),
        "rutherford_keep_mask": rutherford_keep_mask,
        "combined_scan_probability_estimate": (
            combined_scan_probability.astype(np.float32)
        ),
        "combined_scan_probability_cutoff": np.asarray(
            COMBINED_SCAN_PROBABILITY_MIN
        ),
        "combined_scan_high_priority_cutoff": np.asarray(
            COMBINED_SCAN_HIGH_PRIORITY_MIN
        ),
        "combined_scan_keep_mask": combined_scan_keep_mask,
        "combined_scan_high_priority_mask": (
            combined_scan_high_priority_mask
        ),
    }
    filtered_payload.update({
        f"filtered_{name}": value for name, value in filtered_outputs.items()
    })
    np.savez_compressed(RUTHERFORD_FILTERED_RESULT_NPZ, **filtered_payload)
    print("Saved", RUTHERFORD_FILTERED_RESULT_NPZ)


# Full-range heat maps

All maps include contours of the displayed quantity and optional Mathieu
stability-boundary overlays. Points span the complete configured mass and charge
ranges.


In [ ]:
from matplotlib.colors import LogNorm, ListedColormap
from matplotlib.ticker import MaxNLocator
from matplotlib.patches import Patch
from matplotlib.lines import Line2D


def _add_mathieu_boundaries(ax):
    if not SHOW_MATHIEU_BOUNDARIES:
        return
    eps_line = np.geomspace(EPS_MIN, EPS_MAX, 800)
    for index, slope in enumerate(MATHIEU_BOUNDARY_SLOPES_KG, start=1):
        mass_line = slope * eps_line
        visible = (mass_line >= M_DM_MIN_KG) & (mass_line <= M_DM_MAX_KG)
        if np.any(visible):
            ax.plot(
                eps_line[visible], mass_line[visible], linestyle="--",
                linewidth=1.0,
                label="Mathieu stability boundaries" if index == 1 else None,
                zorder=8,
            )


def _levels(data, logarithmic=True, n=8):
    finite = np.asarray(data, dtype=float)
    finite = finite[np.isfinite(finite)]
    if logarithmic:
        finite = finite[finite > 0.0]
    if finite.size == 0 or np.max(finite) <= np.min(finite):
        return np.array([])
    if logarithmic:
        return np.geomspace(np.min(finite), np.max(finite), n + 2)[1:-1]
    ticks = MaxNLocator(nbins=n).tick_values(np.min(finite), np.max(finite))
    return ticks[(ticks > np.min(finite)) & (ticks < np.max(finite))]


def _add_rutherford_probability_contours(ax):
    probability = np.asarray(
        rutherford_probability_on_potential_grid, dtype=float
    )
    finite = probability[np.isfinite(probability)]
    if finite.size == 0:
        return

    threshold = float(RUTHERFORD_RECOIL_PROBABILITY_MIN)
    if np.nanmin(finite) < threshold < np.nanmax(finite):
        cutoff = ax.contour(
            EPS_grid, M_grid, probability,
            levels=[threshold], colors="red", linewidths=1.6,
            linestyles="-", zorder=9,
        )
        ax.clabel(
            cutoff, inline=True, fontsize=7,
            fmt=lambda _: rf"$P_{{R}}={threshold:.0e}$",
        )

    higher = np.asarray(RUTHERFORD_PROBABILITY_CONTOURS, dtype=float)
    higher = higher[
        (higher > threshold)
        & (higher > np.nanmin(finite))
        & (higher < np.nanmax(finite))
    ]
    if higher.size:
        contours = ax.contour(
            EPS_grid, M_grid, probability,
            levels=higher, colors="white", linewidths=0.8,
            linestyles="--", alpha=0.9, zorder=7,
        )
        ax.clabel(
            contours, inline=True, fontsize=6,
            fmt=lambda value: rf"$P_R={value:.0e}$",
        )


def plot_scan_metric(data, title, label, *, probability=False, scale=1.0):
    raw = np.asarray(data, dtype=float) * scale
    z = np.where(rutherford_keep_mask, raw, np.nan)
    finite = z[np.isfinite(z)]
    if finite.size == 0:
        print(
            "No data survive the Rutherford probability cutoff for", title
        )
        return

    fig, ax = plt.subplots(figsize=(9.4, 6.9))

    # Make the excluded portion explicit instead of leaving an unexplained blank.
    excluded = np.where(~rutherford_keep_mask, 1.0, np.nan)
    ax.pcolormesh(
        EPS_grid, M_grid, excluded,
        shading="auto", cmap=ListedColormap(["0.84"]),
        vmin=0.0, vmax=1.0, rasterized=True, zorder=0,
    )

    if probability:
        mesh = ax.pcolormesh(
            EPS_grid, M_grid, np.ma.masked_invalid(z),
            shading="auto", vmin=0.0, vmax=1.0,
            rasterized=True, zorder=2,
        )
        candidates = np.array(
            [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 0.5]
        )
        levels = candidates[
            (candidates > np.min(finite)) & (candidates < np.max(finite))
        ]
    else:
        positive = finite[finite > 0.0]
        norm = (
            LogNorm(vmin=np.min(positive), vmax=np.max(positive))
            if positive.size and np.max(positive) > np.min(positive)
            else None
        )
        mesh = ax.pcolormesh(
            EPS_grid, M_grid, np.ma.masked_invalid(z),
            shading="auto", norm=norm, rasterized=True, zorder=2,
        )
        levels = _levels(z, logarithmic=positive.size > 0)

    if levels.size:
        contour = ax.contour(
            EPS_grid, M_grid, z, levels=levels,
            colors="black", linewidths=0.75, zorder=6,
        )
        ax.clabel(contour, inline=True, fontsize=7, fmt="%.2g")

    _add_rutherford_probability_contours(ax)
    _add_mathieu_boundaries(ax)

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"Charge fraction $\epsilon$")
    ax.set_ylabel(r"DM mass $m_{\rm DM}$ [kg]")
    ax.set_title(
        title
        + "\n"
        + rf"shown only where $P_{{\rm Rutherford}}\geq"
          rf"{RUTHERFORD_RECOIL_PROBABILITY_MIN:.0e}$"
    )
    ax.grid(True, which="both", alpha=0.18)
    fig.colorbar(mesh, ax=ax, label=label)

    handles, labels = ax.get_legend_handles_labels()
    handles.extend([
        Patch(facecolor="0.84", edgecolor="0.55", label=(
            rf"Excluded: $P_{{\rm Rutherford}}<"
            rf"{RUTHERFORD_RECOIL_PROBABILITY_MIN:.0e}$"
        )),
        Line2D([0], [0], color="red", linewidth=1.6, label=(
            rf"Rutherford cutoff $P={RUTHERFORD_RECOIL_PROBABILITY_MIN:.0e}$"
        )),
    ])
    labels.extend([h.get_label() for h in handles[-2:]])
    ax.legend(handles, labels, loc="best", fontsize=7.5, framealpha=0.9)

    fig.tight_layout()
    plt.show()


plot_scan_metric(
    outputs["barrier_access_probability"],
    "MB-weighted probability of passing the potential barrier",
    "Probability", probability=True,
)
plot_scan_metric(
    outputs["ion_upper_probability_ge_threshold"],
    "Probability that the barrier-filtered recoil upper bound exceeds threshold",
    "Probability", probability=True,
)
plot_scan_metric(
    outputs["ion_upper_unconditional_mean_J"],
    "Barrier-filtered mean ion-recoil upper bound",
    "Energy [J per incoming DM]",
)
plot_scan_metric(
    outputs["ion_upper_conditional_median_J"],
    "Conditional median barrier-filtered ion-recoil upper bound",
    "Energy [J]",
)
plot_scan_metric(
    outputs["ion_upper_conditional_p90_J"],
    "Conditional p90 barrier-filtered ion-recoil upper bound",
    "Energy [J]",
)
plot_scan_metric(
    outputs["ion_upper_q10_fullsphere_mean_J"],
    "10th-percentile MB speed: full-sphere mean recoil upper bound",
    "Energy [J per incoming DM]",
)
plot_scan_metric(
    outputs["q10_r_min_threshold_m"],
    "10th-percentile speed: free-space threshold closest approach",
    r"$r_{\min}$ [$\mu$m]", scale=1.0e6,
)


def plot_combined_scan_probability():
    """Show where both free-space recoil and barrier passage are plausible.

    The displayed estimate is

        sum_q w_q P_Rutherford(threshold recoil | q)
                  P_barrier(reach threshold radius | q),

    where q indexes the representative Maxwell--Boltzmann speed bins.  It is a
    screening metric for selecting full trajectory simulations, not an exact
    coupled trap-and-collision probability.
    """
    probability = np.asarray(combined_scan_probability, dtype=float)
    finite_positive = probability[np.isfinite(probability) & (probability > 0.0)]
    if finite_positive.size == 0:
        print("No positive combined scan probabilities are available.")
        return

    fig, ax = plt.subplots(figsize=(9.6, 7.0))

    below = np.where(~combined_scan_keep_mask, 1.0, np.nan)
    ax.pcolormesh(
        EPS_grid, M_grid, below,
        shading="auto", cmap=ListedColormap(["0.86"]),
        vmin=0.0, vmax=1.0, rasterized=True, zorder=0,
    )

    shown = np.where(combined_scan_keep_mask, probability, np.nan)
    shown_positive = shown[np.isfinite(shown) & (shown > 0.0)]
    if shown_positive.size:
        mesh = ax.pcolormesh(
            EPS_grid, M_grid, np.ma.masked_invalid(shown),
            shading="auto",
            norm=LogNorm(
                vmin=max(
                    COMBINED_SCAN_PROBABILITY_MIN,
                    float(np.min(shown_positive)),
                ),
                vmax=float(np.max(shown_positive)),
            ),
            rasterized=True, zorder=2,
        )
        fig.colorbar(
            mesh, ax=ax,
            label=(
                "Speed-matched combined screening probability "
                r"$P_{\rm scan}$"
            ),
        )

    contour_candidates = np.asarray(
        COMBINED_SCAN_PROBABILITY_CONTOURS, dtype=float
    )
    lo = float(np.nanmin(finite_positive))
    hi = float(np.nanmax(finite_positive))
    contour_levels = contour_candidates[
        (contour_candidates > lo) & (contour_candidates < hi)
    ]
    if contour_levels.size:
        contours = ax.contour(
            EPS_grid, M_grid, probability,
            levels=contour_levels, colors="black",
            linewidths=0.85, zorder=6,
        )
        ax.clabel(
            contours, inline=True, fontsize=7,
            fmt=lambda value: rf"$P_{{\rm scan}}={value:.0e}$",
        )

    if lo < COMBINED_SCAN_PROBABILITY_MIN < hi:
        worth = ax.contour(
            EPS_grid, M_grid, probability,
            levels=[COMBINED_SCAN_PROBABILITY_MIN],
            colors="red", linewidths=1.8, zorder=9,
        )
        ax.clabel(
            worth, inline=True, fontsize=8,
            fmt=lambda _: "worth scanning",
        )

    if lo < COMBINED_SCAN_HIGH_PRIORITY_MIN < hi:
        priority = ax.contour(
            EPS_grid, M_grid, probability,
            levels=[COMBINED_SCAN_HIGH_PRIORITY_MIN],
            colors="gold", linewidths=1.8,
            linestyles="--", zorder=9,
        )
        ax.clabel(
            priority, inline=True, fontsize=8,
            fmt=lambda _: "high priority",
        )

    _add_mathieu_boundaries(ax)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"Charge fraction $\epsilon$")
    ax.set_ylabel(r"DM mass $m_{\rm DM}$ [kg]")
    ax.set_title(
        "Recommended parameter space for full trapped-DM simulations\n"
        + rf"worth scanning where $P_{{\rm scan}}\geq"
          rf"{COMBINED_SCAN_PROBABILITY_MIN:.0e}$"
    )
    ax.grid(True, which="both", alpha=0.18)

    handles, labels = ax.get_legend_handles_labels()
    handles.extend([
        Patch(
            facecolor="0.86", edgecolor="0.55",
            label=(
                rf"Lower priority: $P_{{\rm scan}}<"
                rf"{COMBINED_SCAN_PROBABILITY_MIN:.0e}$"
            ),
        ),
        Line2D(
            [0], [0], color="red", linewidth=1.8,
            label=rf"Worth scanning: $P_{{\rm scan}}={COMBINED_SCAN_PROBABILITY_MIN:.0e}$",
        ),
        Line2D(
            [0], [0], color="gold", linewidth=1.8, linestyle="--",
            label=rf"High priority: $P_{{\rm scan}}={COMBINED_SCAN_HIGH_PRIORITY_MIN:.0e}$",
        ),
    ])
    labels.extend([h.get_label() for h in handles[-3:]])
    ax.legend(handles, labels, loc="best", fontsize=7.3, framealpha=0.92)
    fig.tight_layout()
    plt.show()


plot_combined_scan_probability()


# Potential Landscape Test

In [ ]:
m_dm = 2.636651e-25
eps = 2.335721

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm
from scipy.stats import maxwell


# ============================================================
# User settings
# ============================================================

m_dm = 2.636651e-25
eps = 2.335721
T_dm = 300.0

# Edit these if your potential_energy(..., total) function uses different labels.
TERM_NAMES = {
    "dc": "dc",
    "rf": "pseudo_rf",
    "coulomb": "coulomb",
}

def add_scalar_percentiles(row_dict, prefix, arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]

    if arr.size == 0:
        for p in ["p50", "p90", "p95", "p99", "p999", "max"]:
            row_dict[f"{prefix}_{p}"] = np.nan
        return row_dict

    row_dict[f"{prefix}_p50"] = np.percentile(arr, 50)
    row_dict[f"{prefix}_p90"] = np.percentile(arr, 90)
    row_dict[f"{prefix}_p95"] = np.percentile(arr, 95)
    row_dict[f"{prefix}_p99"] = np.percentile(arr, 99)
    row_dict[f"{prefix}_p999"] = np.percentile(arr, 99.9)
    row_dict[f"{prefix}_max"] = np.max(arr)

    return row_dict

# Radial diagnostic radii.
radii_um = np.array([
    100.0,
    150.0,
    200.0,
    300.0,
    500.0,
    750.0,
    1000.0,
    2000.0,
    5000.0,
    10000.0,
    20000.0,
])

radii_m = radii_um * 1e-6

# Use fewer sphere points first because force evaluation can be expensive.
n_sphere = 500

# Speeds to compare against the trap potential/force.
# Five midpoint representatives of equal-probability MB bins.
# The lowest tested speed is therefore the 10th-percentile speed.
speed_quantiles = np.array([
    0.10,
    0.30,
    0.50,
    0.70,
    0.90,
])

speed_cases = deterministic_MB_speeds(T_dm, m_dm, speed_quantiles)

print("Representative equal-probability-bin MB speeds:")
for q, v in zip(speed_quantiles, speed_cases):
    print(f"q={q:6.3f}, v={v:12.6f} m/s")


# ============================================================
# Robust wrappers around your functions
# ============================================================

def as_clean_1d(values):
    arr = np.asarray(values, dtype=float).reshape(-1)
    return arr[np.isfinite(arr)]


def as_force_array(values):
    arr = np.asarray(values, dtype=float)
    arr = np.squeeze(arr)

    if arr.ndim == 1:
        if arr.size == 3:
            arr = arr.reshape(1, 3)
        else:
            arr = arr.reshape(-1, 3)

    if arr.ndim > 2:
        arr = arr.reshape(arr.shape[0], -1)

    if arr.shape[-1] != 3:
        arr = arr.reshape(-1, 3)

    good = np.all(np.isfinite(arr), axis=1)
    return arr[good]


def PE_component_on_sphere(n, radius, m_dm, eps, component):
    term = TERM_NAMES[component]
    xyz = xyz_on_valid_sphere(n, radius)

    values = []
    for row in xyz:
        values.append(potential_energy(row, m_dm, eps, term))

    return as_clean_1d(values)


def F_component_on_sphere(n, radius, m_dm, eps, component):
    term = TERM_NAMES[component]
    xyz = xyz_on_valid_sphere(n, radius)

    values = []
    for row in xyz:
        values.append(force([row], m_dm, eps, term))

    return as_force_array(values)


def PE_point(xyz, m_dm, eps, component):
    term = TERM_NAMES[component]
    return float(potential_energy(np.asarray(xyz, dtype=float), m_dm, eps, term))


def F_point(xyz, m_dm, eps, component):
    term = TERM_NAMES[component]
    raw = force([np.asarray(xyz, dtype=float)], m_dm, eps, term)
    arr = np.asarray(raw, dtype=float).reshape(-1, 3)
    return arr[0]


def kinetic_energy(m_dm, speed):
    return 0.5 * m_dm * speed**2


# ============================================================
# 1. Radial potential and force diagnostics on spheres
# ============================================================

potential_rows = []
force_rows = []
ratio_rows = []

for r_um, r in zip(radii_um, radii_m):
    print(f"Evaluating sphere diagnostics at r = {r_um:g} um")

    valid_fraction = valid_fraction_on_sphere(n_sphere, r)
    print(f"r = {r_um:g} um, valid sphere fraction = {valid_fraction:.3f}")
    U_dc = PE_component_on_sphere(n_sphere, r, m_dm, eps, "dc")
    U_rf = PE_component_on_sphere(n_sphere, r, m_dm, eps, "rf")
    U_coulomb = PE_component_on_sphere(n_sphere, r, m_dm, eps, "coulomb")

    # Use common length just in case one component drops invalid points.
    n_good_U = min(len(U_dc), len(U_rf), len(U_coulomb))
    U_dc = U_dc[:n_good_U]
    U_rf = U_rf[:n_good_U]
    U_coulomb = U_coulomb[:n_good_U]

    U_trap = U_dc + U_rf
    U_total = U_trap + U_coulomb

    potential_terms = {
        "dc": U_dc,
        "rf": U_rf,
        "coulomb": U_coulomb,
        "trap_dc_plus_rf": U_trap,
        "total": U_total,
    }

    for name, U in potential_terms.items():
        row = {
            "r_um": r_um,
            "valid_fraction": valid_fraction,
            "term": name,
            "U_min_J": np.nanmin(U),
            "U_median_J": np.nanmedian(U),
            "U_max_J": np.nanmax(U),
            "U_range_J": np.nanmax(U) - np.nanmin(U),
            "U_abs_max_J": np.nanmax(np.abs(U)),
        }

        row = add_scalar_percentiles(row, "U_J", U)
        potential_rows.append(row)

    F_dc = F_component_on_sphere(n_sphere, r, m_dm, eps, "dc")
    F_rf = F_component_on_sphere(n_sphere, r, m_dm, eps, "rf")
    F_coulomb = F_component_on_sphere(n_sphere, r, m_dm, eps, "coulomb")

    n_good_F = min(len(F_dc), len(F_rf), len(F_coulomb))
    F_dc = F_dc[:n_good_F]
    F_rf = F_rf[:n_good_F]
    F_coulomb = F_coulomb[:n_good_F]

    F_trap = F_dc + F_rf
    F_total = F_trap + F_coulomb

    force_terms = {
        "dc": F_dc,
        "rf": F_rf,
        "coulomb": F_coulomb,
        "trap_dc_plus_rf": F_trap,
        "total": F_total,
    }

    for name, F in force_terms.items():
        F_mag = np.linalg.norm(F, axis=1)

        row = {
            "r_um": r_um,
            "valid_fraction": valid_fraction,
            "term": name,
            "F_min_N": np.nanmin(F_mag),
            "F_median_N": np.nanmedian(F_mag),
            "F_max_N": np.nanmax(F_mag),
        }

        row = add_scalar_percentiles(row, "F_N", F_mag)
        force_rows.append(row)

    U_trap_range = np.nanmax(U_trap) - np.nanmin(U_trap)
    U_trap_abs_max = np.nanmax(np.abs(U_trap))
    F_trap_mag = np.linalg.norm(F_trap, axis=1)
    F_trap_max = np.nanmax(F_trap_mag)
    F_trap_median = np.nanmedian(F_trap_mag)

    for speed in speed_cases:
        K0 = kinetic_energy(m_dm, speed)
        deflection_proxy = F_trap_mag * r / (m_dm * speed**2)

        row = {
            "r_um": r_um,
            "valid_fraction": valid_fraction,
            "speed_m_s": speed,
            "K0_J": K0,
            "U_trap_range_over_K0": U_trap_range / K0,
            "U_trap_absmax_over_K0": U_trap_abs_max / K0,
            "deflection_proxy_max": np.nanmax(deflection_proxy),
            "deflection_proxy_median": np.nanmedian(deflection_proxy),
        }

        row = add_scalar_percentiles(row, "deflection_proxy", deflection_proxy)
        ratio_rows.append(row)


potential_stats_df = pd.DataFrame(potential_rows)
force_stats_df = pd.DataFrame(force_rows)
ratio_stats_df = pd.DataFrame(ratio_rows)

print()
print("Potential radial stats:")
print(potential_stats_df.to_string(index=False))

print()
print("Force radial stats:")
print(force_stats_df.to_string(index=False))

print()
print("Trap ratio stats:")
print(ratio_stats_df.to_string(index=False))


# ============================================================
# 2. Radial plots
# ============================================================

def plot_potential_absmax():
    fig, ax = plt.subplots(figsize=(8, 6))

    for term in ["dc", "rf", "coulomb", "trap_dc_plus_rf", "total"]:
        sub = potential_stats_df[potential_stats_df["term"] == term]
        ax.plot(
            sub["r_um"],
            sub["U_abs_max_J"],
            marker="o",
            label=term,
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"radius [$\mu$m]")
    ax.set_ylabel(r"$\max |U|$ [J]")
    ax.set_title("Potential magnitude vs radius")
    ax.legend()
    ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_potential_range():
    fig, ax = plt.subplots(figsize=(8, 6))

    for term in ["dc", "rf", "coulomb", "trap_dc_plus_rf", "total"]:
        sub = potential_stats_df[potential_stats_df["term"] == term]
        ax.plot(
            sub["r_um"],
            sub["U_range_J"],
            marker="o",
            label=term,
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"radius [$\mu$m]")
    ax.set_ylabel(r"$U_{\max}-U_{\min}$ [J]")
    ax.set_title("Angular potential range vs radius")
    ax.legend()
    ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_force_max():
    fig, ax = plt.subplots(figsize=(8, 6))

    for term in ["dc", "rf", "coulomb", "trap_dc_plus_rf", "total"]:
        sub = force_stats_df[force_stats_df["term"] == term]
        ax.plot(
            sub["r_um"],
            sub["F_max_N"],
            marker="o",
            label=term,
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"radius [$\mu$m]")
    ax.set_ylabel(r"$\max |F|$ [N]")
    ax.set_title("Force magnitude vs radius")
    ax.legend()
    ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_U_range_over_K0():
    fig, ax = plt.subplots(figsize=(8, 6))

    for speed in speed_cases:
        sub = ratio_stats_df[np.isclose(ratio_stats_df["speed_m_s"], speed)]
        ax.plot(
            sub["r_um"],
            sub["U_trap_range_over_K0"],
            marker="o",
            label=f"{speed:.1f} m/s",
        )

    ax.axhline(1e-2, linestyle="--", label="1%")
    ax.axhline(5e-2, linestyle=":", label="5%")

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"radius [$\mu$m]")
    ax.set_ylabel(r"$(U_{\max}-U_{\min})/K_0$")
    ax.set_title("Trap potential range relative to DM kinetic energy")
    ax.legend(fontsize=8)
    ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_deflection_proxy():
    fig, ax = plt.subplots(figsize=(8, 6))

    for speed in speed_cases:
        sub = ratio_stats_df[np.isclose(ratio_stats_df["speed_m_s"], speed)]
        ax.plot(
            sub["r_um"],
            sub["deflection_proxy_max"],
            marker="o",
            label=f"{speed:.1f} m/s",
        )

    ax.axhline(1e-2, linestyle="--", label="1%")
    ax.axhline(5e-2, linestyle=":", label="5%")

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"radius [$\mu$m]")
    ax.set_ylabel(r"$|F_{\rm trap}| r /(m v^2)$")
    ax.set_title("Trap deflection proxy vs radius")
    ax.legend(fontsize=8)
    ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_potential_absmax()
plot_potential_range()
plot_force_max()
plot_U_range_over_K0()
plot_deflection_proxy()


# ============================================================
# 3. 2D potential and force maps
# ============================================================

def make_plane_grid(plane="xz", extent_um=500.0, n_grid=121, fixed_um=0.0):
    vals_um = np.linspace(-extent_um, extent_um, n_grid)
    A_um, B_um = np.meshgrid(vals_um, vals_um, indexing="xy")

    A = A_um * 1e-6
    B = B_um * 1e-6
    fixed = fixed_um * 1e-6

    points = np.zeros((n_grid, n_grid, 3))

    if plane == "xy":
        points[..., 0] = A
        points[..., 1] = B
        points[..., 2] = fixed
        xlabel = r"$x$ [$\mu$m]"
        ylabel = r"$y$ [$\mu$m]"

    elif plane == "xz":
        points[..., 0] = A
        points[..., 1] = fixed
        points[..., 2] = B
        xlabel = r"$x$ [$\mu$m]"
        ylabel = r"$z$ [$\mu$m]"

    elif plane == "yz":
        points[..., 0] = fixed
        points[..., 1] = A
        points[..., 2] = B
        xlabel = r"$y$ [$\mu$m]"
        ylabel = r"$z$ [$\mu$m]"

    else:
        raise ValueError("plane must be 'xy', 'xz', or 'yz'")

    return A_um, B_um, points, xlabel, ylabel


def evaluate_plane_potential(points, component):
    n1, n2, _ = points.shape
    U = np.full((n1, n2), np.nan)

    for i in range(n1):
        for j in range(n2):
            xyz = points[i, j]

            if not valid_xyz_mask(xyz):
                continue

            U[i, j] = PE_point(xyz, m_dm, eps, component)

    return U


def evaluate_plane_force(points, component):
    n1, n2, _ = points.shape
    F = np.full((n1, n2, 3), np.nan)

    for i in range(n1):
        for j in range(n2):
            xyz = points[i, j]

            if not valid_xyz_mask(xyz):
                continue

            F[i, j] = F_point(xyz, m_dm, eps, component)

    return F


def plot_plane_potential(plane="xz", extent_um=500.0, n_grid=121):
    A_um, B_um, points, xlabel, ylabel = make_plane_grid(
        plane=plane,
        extent_um=extent_um,
        n_grid=n_grid,
    )

    print(f"Evaluating 2D potential map in {plane} plane")

    U_dc = evaluate_plane_potential(points, "dc")
    U_rf = evaluate_plane_potential(points, "rf")
    U_coulomb = evaluate_plane_potential(points, "coulomb")

    maps = {
        "dc": U_dc,
        "rf": U_rf,
        "coulomb": U_coulomb,
        "trap_dc_plus_rf": U_dc + U_rf,
        "total": U_dc + U_rf + U_coulomb,
    }

    for name, U in maps.items():
        finite_U = U[np.isfinite(U)]
        if finite_U.size == 0:
            print(f"Skipping {name}; no finite values")
            continue

        vmax = np.nanmax(np.abs(finite_U))
        if vmax == 0:
            vmax = 1.0

        fig, ax = plt.subplots(figsize=(7, 6))

        norm = SymLogNorm(
            linthresh=max(vmax * 1e-4, 1e-30),
            vmin=-vmax,
            vmax=vmax,
        )

        pcm = ax.pcolormesh(
            A_um,
            B_um,
            U,
            shading="auto",
            norm=norm,
        )

        ax.contour(
            A_um,
            B_um,
            U,
            levels=15,
            linewidths=0.7,
        )

        ax.set_aspect("equal")
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(f"{name} potential in {plane} plane")

        cbar = fig.colorbar(pcm, ax=ax)
        cbar.set_label("Potential energy [J]")

        if plane in ["xz", "yz"]:
            ax.axhline(z_min_valid * 1e6, linestyle="--", linewidth=2, label="validity boundary z = -70 um")
            ax.legend()
        plt.tight_layout()
        plt.show()


def plot_plane_force(plane="xz", extent_um=500.0, n_grid=81, stride=5):
    A_um, B_um, points, xlabel, ylabel = make_plane_grid(
        plane=plane,
        extent_um=extent_um,
        n_grid=n_grid,
    )

    print(f"Evaluating 2D force map in {plane} plane")

    F_dc = evaluate_plane_force(points, "dc")
    F_rf = evaluate_plane_force(points, "rf")
    F_coulomb = evaluate_plane_force(points, "coulomb")

    F_trap = F_dc + F_rf
    F_total = F_trap + F_coulomb

    force_maps = {
        "trap_dc_plus_rf": F_trap,
        "total": F_total,
    }

    for name, F in force_maps.items():
        F_mag = np.linalg.norm(F, axis=2)

        if plane == "xy":
            FA = F[..., 0]
            FB = F[..., 1]
        elif plane == "xz":
            FA = F[..., 0]
            FB = F[..., 2]
        elif plane == "yz":
            FA = F[..., 1]
            FB = F[..., 2]

        fig, ax = plt.subplots(figsize=(7, 6))

        pcm = ax.pcolormesh(
            A_um,
            B_um,
            F_mag,
            shading="auto",
        )

        ax.quiver(
            A_um[::stride, ::stride],
            B_um[::stride, ::stride],
            FA[::stride, ::stride],
            FB[::stride, ::stride],
        )

        ax.set_aspect("equal")
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(f"{name} force field in {plane} plane")

        cbar = fig.colorbar(pcm, ax=ax)
        cbar.set_label(r"$|F|$ [N]")

        if plane in ["xz", "yz"]:
            ax.axhline(z_min_valid * 1e6, linestyle="--", linewidth=2, label="validity boundary z = -70 um")
            ax.legend()
        plt.tight_layout()
        plt.show()


# ============================================================
# 4. Plot all important 2D planes
# ============================================================

planes_to_plot = ["xz", "xy", "yz"]

for plane in planes_to_plot:
    plot_plane_potential(
        plane=plane,
        extent_um=500.0,
        n_grid=121,
    )

    plot_plane_force(
        plane=plane,
        extent_um=500.0,
        n_grid=81,
        stride=5,
    )

def plot_deflection_proxy_percentiles(speed_to_plot=192.788408):
    speeds_unique = np.sort(ratio_stats_df["speed_m_s"].unique())
    speed_actual = speeds_unique[np.argmin(np.abs(speeds_unique - speed_to_plot))]

    sub = ratio_stats_df[np.isclose(ratio_stats_df["speed_m_s"], speed_actual)]

    fig, ax = plt.subplots(figsize=(8, 6))

    for col, label in [
        ("deflection_proxy_p50", "p50"),
        ("deflection_proxy_p90", "p90"),
        ("deflection_proxy_p95", "p95"),
        ("deflection_proxy_p99", "p99"),
        ("deflection_proxy_p999", "p99.9"),
        ("deflection_proxy_max", "max"),
    ]:
        ax.plot(sub["r_um"], sub[col], marker="o", label=label)

    ax.axhline(1e-2, linestyle="--", label="1%")
    ax.axhline(5e-2, linestyle=":", label="5%")

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"radius [$\mu$m]")
    ax.set_ylabel(r"$|F_{\rm trap}| r /(m v^2)$")
    ax.set_title(f"Deflection proxy percentiles, v = {speed_actual:.1f} m/s")
    ax.legend()
    ax.grid(True, which="both", alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_deflection_proxy_percentiles(speed_to_plot=speed_cases[0])
plot_deflection_proxy_percentiles(speed_to_plot=speed_cases[2])
plot_deflection_proxy_percentiles(speed_to_plot=speed_cases[-1])
